<a href="https://colab.research.google.com/github/wallynovak/nematode_aggregate_analysis/blob/main/Aggregate_Counter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img align="right" src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/PyBMB_logo.png" width="150" height="150" />

# Molecules to Medicine Summer Camp
## Counting Fluorescent Protein Aggregates in *C. elegans*

**Purpose:**
In this notebook you will use Python to automatically count fluorescent
protein aggregates in microscope images of *C. elegans* worms. This is
the same kind of image analysis that scientists use to study diseases
like Alzheimer's and Parkinson's — where misfolded proteins clump
together into toxic aggregates.

You will learn to:
- Load and prepare a fluorescence microscope image for analysis
- Draw a region of interest around your worm and a background region
- Set a brightness threshold to separate aggregates from background
- Automatically count individual aggregates using image segmentation
- Review, verify, and export your results

**What you need:**
A fluorescence microscope image of a *C. elegans* worm expressing
a fluorescently tagged protein, saved as a JPEG, PNG, or TIFF file.

**Libraries:**
* `scikit-image` — image loading, filtering, and segmentation
* `scipy` — distance transforms and watershed segmentation
* `numpy` — numerical arrays and calculations
* `matplotlib` — displaying images and plots
* `pandas` — organizing results in a table
* `PIL` — image format conversion for the dashboard
* `ipywidgets` — interactive sliders and controls

**Status:** Molecules to Medicine Summer Camp — 2025

**License**

<img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/by-nc-sa.png" width="100"/>
CC BY-NC-SA — free to use and adapt for non-commercial educational purposes with attribution.

---
**Authorship:** Molecules to Medicine / CodeBMB Team

**Acknowledgements:** This notebook is supported by NSF IUSE 2518733.

**Contact Info:** codingBMB@gmail.com

# 0. How to Run This Notebook

To run a cell, click the ▶ play button on the left side of the cell.

![run button image](https://github.com/wallynovak/biochemistry_seq_analysis/blob/main/images/run.png?raw=1)

A cell is still running if you see a stop button with a moving circle.
A completed cell shows a number in brackets (e.g. [1]) and a checkmark.

**Always run the cells in order from top to bottom.**

> ⚠️ If you restart the notebook at any point, run all cells again
> from the beginning — Python forgets everything when it restarts.

# 1. Setting Up

## Libraries Used in This Notebook

| Library | Purpose |
|---|---|
| `scikit-image` | Image loading, thresholding, segmentation, and measurement |
| `scipy` | Distance transforms used in the watershed algorithm |
| `numpy` | Fast numerical calculations on image pixel arrays |
| `matplotlib` | Displaying images and plots |
| `pandas` | Organizing aggregate measurements in a table |
| `PIL` | Converting images for display in the interactive dashboard |
| `ipywidgets` | Interactive sliders, buttons, and dropdown menus |

Run the cell below first — everything else in this notebook depends on it.
This may take about 30 seconds the first time.

In [ ]:
# Run this cell to load all required tools
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
import base64
import io
import IPython

from skimage import io as sk_io, filters, morphology, segmentation, measure, color
from skimage.util import img_as_ubyte
from skimage.feature import peak_local_max
from skimage.color import label2rgb
from skimage.measure import find_contours
from scipy import ndimage
from PIL import Image

from ipywidgets import (IntSlider, FloatSlider, Output, VBox, HBox,
                        Button, Dropdown, Layout, HTML)
from IPython.display import display, clear_output
from google.colab import files, output

print("✅ All libraries loaded successfully!")
print("You are ready to begin.")

# 2. Background: Why Are We Counting Aggregates?

## *C. elegans* as a Disease Model

*C. elegans* is a tiny, transparent roundworm about 1 mm long.
Scientists love it because:
- You can **see inside it** with a microscope — it's completely transparent
- It shares many of the **same genes as humans**
- It grows from egg to adult in only **3 days**
- It is inexpensive and easy to work with in large numbers

By engineering worms to produce human disease proteins, scientists
can study diseases like **Alzheimer's**, **Parkinson's**, and
**Huntington's disease** in a living animal — quickly and inexpensively.

---

## What Are Protein Aggregates?

In healthy cells, proteins fold into specific 3D shapes to do their jobs.
In some diseases, proteins **misfold** and clump together into
**aggregates** — toxic tangles that can damage or kill cells.

In this experiment, your worms express a **fluorescently tagged protein**
that forms visible aggregates. In the fluorescence microscope, aggregates
appear as **bright spots** against a darker background.

---

## Why Count Them with Code?

Counting aggregates by eye is:
- **Slow** — each worm can have dozens to hundreds of spots
- **Inconsistent** — different people count differently
- **Biased** — you might unconsciously count differently in treated vs. control worms

A Python program counts every pixel the same way every time,
making the results **objective** and **reproducible** — two of the
most important qualities in scientific data.

---

## How This Notebook Works

| Step | What we do | Why |
|---|---|---|
| **1** | Load the image and extract the green fluorescence channel | Prepare the image for analysis |
| **2** | Draw a box around the worm and a background region | Exclude the scale bar and other artifacts |
| **3** | Set a brightness threshold | Separate bright aggregates from dim background |
| **4** | Apply watershed segmentation | Split touching aggregates into individuals |
| **5** | Inspect the detected aggregates with sliders | Verify the computer found real objects |
| **6** | Use the classification dashboard | Accept, flag, or reject each detection |
| **7** | Export the data | Save your count for analysis |

# Step 1 — Load and Prepare Your Image

## Uploading Your Image

When you run the cell below, a **Choose Files** button will appear.
Click it and select your fluorescence microscope image from your computer.

## What the Code Does

1. **Uploads** your image file into this Colab session
2. **Extracts the green channel** —  in this experiment, the protein aggregates are fused to fluorescent proteins that emit in the green spectra, so the green channel carries the signal we care about. The red and blue channels mostly contain background noise
3. **Converts to 16-bit format** — your camera saves images with
   256 brightness levels (8-bit). Converting to 16-bit gives us
   65,536 levels, allowing more precise detection of dim aggregates
4. **Displays the prepared image** so you can check it loaded correctly

> ⚠️ If your image appears completely black, the green channel may
> not contain your signal. Ask your instructor whether your dye
> emits in a different color channel.

In [ ]:
# Step 1: Upload your image, extract the green channel, and convert to 16-bit
# Click the 'Choose Files' button that appears after running this cell

print("Please select your fluorescence microscope image:")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No image was uploaded. Please run this cell again and select a file.")

# Load the first uploaded file
for filename, content in uploaded.items():
    image_data   = io.BytesIO(content)
    raw_image    = sk_io.imread(image_data)
    print(f"✅ Loaded: {filename}")
    print(f"   Image dimensions: {raw_image.shape}")
    break

# Store the original color image for later display in the dashboard
original_rgb_img = raw_image.copy()

# Extract the green channel (index 1 in an RGB image)
# Fluorescent proteins emit primarily in the green wavelength range
if raw_image.ndim == 3:
    img = raw_image[:, :, 1]   # green channel
    print("   Green channel extracted for analysis.")
else:
    img = raw_image             # already grayscale
    print("   Image is already grayscale.")

# Convert to 16-bit to increase precision
# 8-bit  = 256 possible brightness values
# 16-bit = 65,536 possible brightness values
img_16bit = img.astype(np.uint16)

# Display the prepared image
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(original_rgb_img)
axes[0].set_title('Original Color Image', fontsize=13)
axes[0].axis('off')

axes[1].imshow(img_16bit, cmap='gray')
axes[1].set_title('Green Channel (16-bit) — used for analysis', fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print()
print(f"Image size:       {img_16bit.shape[1]} × {img_16bit.shape[0]} pixels")
print(f"Brightness range: {img_16bit.min()} to {img_16bit.max()}")
print()
print("✅ Image prepared. Proceed to Step 2.")

# Step 2 — Define Your Analysis Regions

## Why Do We Need to Draw Boxes?

Your image contains more than just the worm:
- A **scale bar** (usually a bright white line in a corner)
- Areas of **empty background** away from the worm
- Possibly parts of **other worms** or debris

If we analyze the entire image, the scale bar will be detected as
a cluster of aggregates — giving us the wrong count.

By drawing a box around just the worm, we tell the computer
exactly where to look.

## The Two Boxes You Will Draw

**🔵 Worm box (cyan):**
Draw this tightly around your worm. Everything outside this box
will be ignored during analysis.

**🟡 Background box (yellow):**
Draw this in an empty area of the image — no worm, no scale bar,
no bright spots. This gives the computer a sample of what the
true background looks like, so it can correctly distinguish
dim background from real aggregates.

## How to Use the Sliders

1. Look at the reference image that appears — it has a grid to help
   you read pixel coordinates
2. Move the sliders to position each box
3. Check the live preview to confirm your boxes look correct
4. Click **✅ Confirm ROI Selection** when satisfied

> 💡 **Tip:** The right panel shows a zoomed crop of your worm box
> so you can see exactly what will be analyzed.

In [ ]:
# Step 2: Draw boxes around the worm and a background region
# Use the sliders to position each box, then click Confirm

img_h, img_w = img_16bit.shape

# ── Step 2a: Show reference image with grid ────────────────
def nice_spacing(dim, n=10):
    raw = dim / n
    for base in [500, 200, 100, 50, 20, 10, 5]:
        if raw >= base:
            return base
    return 5

grid_spacing = nice_spacing(min(img_w, img_h))

fig_ref, ax_ref = plt.subplots(figsize=(10, 8))
ax_ref.imshow(img_16bit, cmap='gray')

for x in range(0, img_w + 1, grid_spacing):
    ax_ref.axvline(x=x, color='white', alpha=0.35, linewidth=0.7)
for y in range(0, img_h + 1, grid_spacing):
    ax_ref.axhline(y=y, color='white', alpha=0.35, linewidth=0.7)

ax_ref.set_xticks(range(0, img_w + 1, grid_spacing))
ax_ref.set_yticks(range(0, img_h + 1, grid_spacing))
ax_ref.tick_params(axis='both', colors='black', labelsize=8)
for spine in ax_ref.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1)

ax_ref.set_title(
    f'Reference Image — use the grid to estimate pixel coordinates\n'
    f'Grid spacing: {grid_spacing} px (square grid)',
    fontsize=12, color='black'
)
ax_ref.set_xlabel('x  (pixel column)', fontsize=10, color='black')
ax_ref.set_ylabel('y  (pixel row)',    fontsize=10, color='black')
plt.tight_layout()
plt.show()

print(f"Image: {img_w} px wide × {img_h} px tall")
print("Use the sliders below to draw your boxes, then click Confirm.")
print()

# ── Step 2b: Create sliders ────────────────────────────────
STEP = max(1, img_w // 200)

worm_x1 = IntSlider(value=int(img_w*0.10), min=0, max=img_w-1, step=STEP,
                     description='x1 (left):',  style={'description_width':'110px'},
                     layout=Layout(width='450px'))
worm_x2 = IntSlider(value=int(img_w*0.90), min=0, max=img_w-1, step=STEP,
                     description='x2 (right):', style={'description_width':'110px'},
                     layout=Layout(width='450px'))
worm_y1 = IntSlider(value=int(img_h*0.10), min=0, max=img_h-1, step=STEP,
                     description='y1 (top):',   style={'description_width':'110px'},
                     layout=Layout(width='450px'))
worm_y2 = IntSlider(value=int(img_h*0.90), min=0, max=img_h-1, step=STEP,
                     description='y2 (bottom):',style={'description_width':'110px'},
                     layout=Layout(width='450px'))
bg_x1 = IntSlider(value=0,               min=0, max=img_w-1, step=STEP,
                   description='x1 (left):',  style={'description_width':'110px'},
                   layout=Layout(width='450px'))
bg_x2 = IntSlider(value=int(img_w*0.15), min=0, max=img_w-1, step=STEP,
                   description='x2 (right):', style={'description_width':'110px'},
                   layout=Layout(width='450px'))
bg_y1 = IntSlider(value=0,               min=0, max=img_h-1, step=STEP,
                   description='y1 (top):',   style={'description_width':'110px'},
                   layout=Layout(width='450px'))
bg_y2 = IntSlider(value=int(img_h*0.15), min=0, max=img_h-1, step=STEP,
                   description='y2 (bottom):',style={'description_width':'110px'},
                   layout=Layout(width='450px'))

import matplotlib.patches as patches
plot_output  = Output()
stats_output = Output()

def update_preview(change=None):
    wx1,wx2 = worm_x1.value, worm_x2.value
    wy1,wy2 = worm_y1.value, worm_y2.value
    bx1,bx2 = bg_x1.value,  bg_x2.value
    by1,by2 = bg_y1.value,  bg_y2.value

    with plot_output:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(img_16bit, cmap='gray')
        if wx2 > wx1 and wy2 > wy1:
            axes[0].add_patch(patches.Rectangle((wx1,wy1), wx2-wx1, wy2-wy1,
                linewidth=2.5, edgecolor='cyan', facecolor='cyan', alpha=0.08))
            axes[0].text(wx1+6, wy1+20, 'WORM', color='cyan', fontsize=11, fontweight='bold')
        if bx2 > bx1 and by2 > by1:
            axes[0].add_patch(patches.Rectangle((bx1,by1), bx2-bx1, by2-by1,
                linewidth=2.5, edgecolor='yellow', facecolor='yellow', alpha=0.08))
            axes[0].text(bx1+6, by1+20, 'BG', color='yellow', fontsize=11, fontweight='bold')
        axes[0].set_title('Full image — both regions', fontsize=11)
        axes[0].axis('off')
        if wx2 > wx1 and wy2 > wy1:
            worm_crop = img_16bit[wy1:wy2, wx1:wx2]
            axes[1].imshow(worm_crop, cmap='gray')
            axes[1].set_title(f'Worm ROI preview\nx:{wx1}–{wx2}  y:{wy1}–{wy2}\n'
                              f'Size: {wx2-wx1} × {wy2-wy1} px', fontsize=10)
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

    with stats_output:
        clear_output(wait=True)
        if bx2 > bx1 and by2 > by1:
            bg_region = img_16bit[by1:by2, bx1:bx2]
            print(f"Background region: {bx2-bx1} × {by2-by1} px")
            print(f"  Mean intensity:  {np.mean(bg_region):.1f}")
            print(f"  Std deviation:   {np.std(bg_region):.1f}")
            worm_mean = np.mean(img_16bit[wy1:wy2, wx1:wx2]) if (wx2>wx1 and wy2>wy1) else 0
            if np.mean(bg_region) > worm_mean * 0.8:
                print("⚠️  Warning: background region looks bright.")
                print("   Choose an area with low, uniform intensity.")
            else:
                print("✅ Background region looks reasonable.")

for s in [worm_x1, worm_x2, worm_y1, worm_y2, bg_x1, bg_x2, bg_y1, bg_y2]:
    s.observe(update_preview, names='value')

confirm_btn = Button(description='✅ Confirm ROI Selection',
                     button_style='success',
                     layout=Layout(width='260px', height='40px'))

def on_confirm(b):
    global img_16bit_roi, mean_background_intensity, std_background_intensity
    global worm_roi_coords, bg_roi_coords
    wx1,wx2 = worm_x1.value, worm_x2.value
    wy1,wy2 = worm_y1.value, worm_y2.value
    bx1,bx2 = bg_x1.value,  bg_x2.value
    by1,by2 = bg_y1.value,  bg_y2.value

    errors = []
    if wx2 <= wx1: errors.append("Worm x2 must be greater than x1")
    if wy2 <= wy1: errors.append("Worm y2 must be greater than y1")
    if bx2 <= bx1: errors.append("Background x2 must be greater than x1")
    if by2 <= by1: errors.append("Background y2 must be greater than y1")
    if (wx2-wx1)*(wy2-wy1) < 100: errors.append("Worm ROI is too small")
    if (bx2-bx1)*(by2-by1) < 100: errors.append("Background ROI is too small")
    if errors:
        for e in errors: print(f"❌ {e}")
        return

    worm_roi_coords = (wx1, wy1, wx2, wy2)
    bg_roi_coords   = (bx1, by1, bx2, by2)

    # Create masked image — zeros outside the worm box
    # This prevents scale bar and other features from being counted
    img_16bit_roi = np.zeros_like(img_16bit)
    img_16bit_roi[wy1:wy2, wx1:wx2] = img_16bit[wy1:wy2, wx1:wx2]

    # Local background from the user-defined background region
    bg_region = img_16bit[by1:by2, bx1:bx2]
    mean_background_intensity = float(np.mean(bg_region))
    std_background_intensity  = float(np.std(bg_region))

    print("=" * 55)
    print("✅ ROI confirmed!")
    print(f"  Worm ROI:  x={wx1}–{wx2}, y={wy1}–{wy2}  ({wx2-wx1}×{wy2-wy1} px)")
    print(f"  BG ROI:    x={bx1}–{bx2}, y={by1}–{by2}  ({bx2-bx1}×{by2-by1} px)")
    print(f"  Mean background intensity: {mean_background_intensity:.2f}")
    print(f"  Std background intensity:  {std_background_intensity:.2f}")
    print("=" * 55)
    print("▶ Run Step 3 to continue.")

    fig_c, axes_c = plt.subplots(1, 2, figsize=(12, 5))
    axes_c[0].imshow(img_16bit, cmap='gray')
    axes_c[0].add_patch(patches.Rectangle((wx1,wy1), wx2-wx1, wy2-wy1,
        linewidth=2.5, edgecolor='cyan', facecolor='none'))
    axes_c[0].add_patch(patches.Rectangle((bx1,by1), bx2-bx1, by2-by1,
        linewidth=2.5, edgecolor='yellow', facecolor='none'))
    axes_c[0].set_title('Original with confirmed ROIs', fontsize=11)
    axes_c[0].axis('off')
    axes_c[1].imshow(img_16bit_roi, cmap='gray')
    axes_c[1].set_title('Worm ROI only — ready for analysis', fontsize=11)
    axes_c[1].axis('off')
    plt.tight_layout()
    plt.show()

confirm_btn.on_click(on_confirm)

worm_box = VBox([
    HTML('<b style="color:cyan; font-size:13px;">🔵 Worm Region (cyan box)</b>'),
    HTML('<i style="font-size:11px; color:#555;">Draw tightly around the worm.</i>'),
    HBox([worm_x1, worm_x2]),
    HBox([worm_y1, worm_y2]),
])
bg_box = VBox([
    HTML('<b style="color:#cccc00; font-size:13px;">🟡 Background Region (yellow box)</b>'),
    HTML('<i style="font-size:11px; color:#555;">Choose an empty area — no worm, no scale bar.</i>'),
    HBox([bg_x1, bg_x2]),
    HBox([bg_y1, bg_y2]),
])
display(VBox([worm_box, HTML('<br>'), bg_box, HTML('<br>'),
              confirm_btn, HTML('<br>'), stats_output, plot_output]))
update_preview()

# Step 3 — Set the Detection Threshold

## What Is a Threshold?

Your image contains pixels of many different brightness levels.
A **threshold** draws a line between:
- Pixels **above** the threshold → bright → aggregate (shown in red)
- Pixels **below** the threshold → dim → background (shown in gray)

Setting the threshold correctly is the most important step in the analysis.

## How to Use the Controls

**Method dropdown:** The notebook offers five automatic thresholding methods.
**Yen** is a good starting point for fluorescence images.

**Threshold slider:** Fine-tune the value manually.
- Move the slider **right** (higher value) to detect fewer, brighter spots
- Move the slider **left** (lower value) to detect more, dimmer spots

## What to Look For

✅ **Good threshold:** Red regions cover the bright aggregate spots
   but not the dim background or worm body.

⚠️ **Too low:** Large regions of the worm body turn red — you're
   detecting everything, including things that aren't aggregates.

⚠️ **Too high:** Nothing or very few spots turn red — you're
   missing real aggregates.

> 💡 There is no single "correct" answer — use your scientific judgment
> and compare your result to what you see in the original image.

In [ ]:
# Step 3: Set the detection threshold
# Adjust the slider until the red regions match the bright aggregate spots

img_min = img_16bit_roi.min()
img_max = img_16bit_roi.max()

# Available automatic thresholding methods
threshold_methods = {
    'Yen':      filters.threshold_yen,
    'Otsu':     filters.threshold_otsu,
    'Li':       filters.threshold_li,
    'Isodata':  filters.threshold_isodata,
    'Triangle': filters.threshold_triangle,
}

method_dropdown = Dropdown(
    options=list(threshold_methods.keys()),
    value='Yen',
    description='Method:',
    layout=Layout(width='200px')
)

def calculate_threshold(method_name):
    try:
        return threshold_methods[method_name](img_16bit_roi)
    except Exception:
        return filters.threshold_otsu(img_16bit_roi)

current_thresh_val = calculate_threshold('Yen')
print(f"Yen's suggested threshold: {current_thresh_val:.0f}  "
      f"(image range: {img_min} – {img_max})")

# Set up the side-by-side plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(img_16bit_roi, cmap='gray')
axes[0].set_title('Worm ROI (green channel)')
axes[0].axis('off')

initial_thresholded = img_16bit_roi > int(current_thresh_val)
normalized = img_16bit_roi.astype(float) / max(img_max, 1)
overlay_initial = color.gray2rgb(img_as_ubyte(normalized))
overlay_initial[initial_thresholded] = [255, 0, 0]
im_after = axes[1].imshow(overlay_initial)
axes[1].set_title(f'After threshold (red = detected, value={int(current_thresh_val)})')
axes[1].axis('off')
plt.tight_layout()

thresh_plot_output = Output()
thresh_text_output = Output()

with thresh_plot_output:
    display(fig)
    plt.close(fig)

global thresh_val, thresholded
thresh_val   = current_thresh_val
thresholded  = img_16bit_roi > thresh_val

def update_threshold(change=None, manual_val=None):
    global thresh_val, thresholded
    if manual_val is not None:
        thresh_val = manual_val
        method_label = "Manual"
    else:
        method_label = method_dropdown.value
        thresh_val   = calculate_threshold(method_label)

    thresholded = img_16bit_roi > thresh_val

    with thresh_text_output:
        clear_output(wait=True)
        print(f"Method: {method_label}   |   Threshold value: {thresh_val:.0f}")

    normalized_cur = img_16bit_roi.astype(float) / max(img_max, 1)
    overlay_cur = color.gray2rgb(img_as_ubyte(normalized_cur))
    overlay_cur[thresholded] = [255, 0, 0]
    im_after.set_data(overlay_cur)
    axes[1].set_title(f'After threshold (red = detected, value={thresh_val:.0f})')

    with thresh_plot_output:
        clear_output(wait=True)
        display(fig)

threshold_slider = IntSlider(
    value=int(current_thresh_val), min=int(img_min), max=int(img_max),
    step=1, description='Threshold:', continuous_update=True,
    layout=Layout(width='500px')
)
inc_btn = Button(description='+', layout=Layout(width='50px'))
dec_btn = Button(description='-', layout=Layout(width='50px'))

def on_inc(b):
    threshold_slider.value = min(threshold_slider.max, threshold_slider.value + 1)
    update_threshold(manual_val=threshold_slider.value)
def on_dec(b):
    threshold_slider.value = max(threshold_slider.min, threshold_slider.value - 1)
    update_threshold(manual_val=threshold_slider.value)

inc_btn.on_click(on_inc)
dec_btn.on_click(on_dec)
threshold_slider.observe(lambda c: update_threshold(manual_val=c.new), names='value')

def on_method_change(change):
    new_val = calculate_threshold(change.new)
    threshold_slider.value = int(new_val)
    update_threshold(change=change)

method_dropdown.observe(on_method_change, names='value')

controls = VBox([
    method_dropdown,
    HBox([dec_btn, threshold_slider, inc_btn]),
    thresh_text_output
])
display(controls, thresh_plot_output)
update_threshold()

# Step 4 — Detect Individual Aggregates

## From Threshold to Count

After thresholding, we have a **binary image** — pixels are either
"on" (aggregate) or "off" (background). But there is a problem:
aggregates that are close together appear as one fused blob.

## The Watershed Algorithm

**Watershed** is a clever technique borrowed from geography.
Imagine the binary image as a landscape where:
- Bright blobs are **hills**
- The algorithm finds the **peak** of each hill (the center of each aggregate)
- It draws **boundaries** between neighboring peaks

This separates touching aggregates into individuals so each one
can be counted and measured separately.

## What You Will See

After running this cell you will see two images:
- **Left:** The binary image (white = above threshold)
- **Right:** The watershed result — each colored region is one
  detected object, colored differently to show separation

> The number shown in the right panel title is the **total number
> of detected regions** — this includes some objects that may not
> be real aggregates (too small, too large, or odd-shaped).
> We will filter these in Steps 5 and 6.

In [ ]:
# Step 4: Convert to binary and run watershed segmentation
# This separates touching aggregates into individual objects

# Binary image: True = above threshold, False = background
binary = thresholded.copy()

# Watershed algorithm:
# 1. Calculate how far each 'on' pixel is from the nearest 'off' pixel
distance = ndimage.distance_transform_edt(binary)

# 2. Find local intensity peaks — each peak is likely one aggregate center
#    min_distance=4 means peaks must be at least 4 pixels apart
coords = peak_local_max(distance, min_distance=4, labels=binary)
local_max_mask = np.zeros(distance.shape, dtype=bool)
local_max_mask[tuple(coords.T)] = True

# 3. Label each peak and use watershed to draw boundaries
markers        = measure.label(local_max_mask)
watershed_result = segmentation.watershed(-distance, markers, mask=binary)

# Measure properties of each detected region
props = measure.regionprops(watershed_result, intensity_image=img_16bit_roi)

# Display results side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(binary, cmap='binary_r')
axes[0].set_title('Binary Image\n(white = above threshold)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(label2rgb(watershed_result, bg_label=0))
axes[1].set_title(f'Watershed Segmentation\n{len(props)} regions detected', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"✅ Watershed complete: {len(props)} regions detected.")
print("   These include real aggregates as well as noise and artifacts.")
print("   Proceed to Step 5 to filter and verify.")

# Step 5 — Inspect Detected Aggregates

## Filtering by Size and Shape

Not all detected regions are real aggregates. Some are:
- **Too small** — single bright pixels (noise)
- **Too large** — parts of the worm body or merged blobs
- **Wrong shape** — elongated streaks rather than round spots

This step lets you filter the detections using **four sliders**:

| Slider | What it controls |
|---|---|
| **Min Size** | Minimum area in pixels — removes tiny noise spots |
| **Max Size** | Maximum area in pixels — removes large non-aggregate regions |
| **Min Circularity** | Shape filter — 1.0 = perfect circle, lower = more irregular |
| **Max Circularity** | Upper circularity limit |

## What to Look For

Adjust the sliders until the **cyan outlines** cover the bright aggregate
spots in the original image but not the worm body, background, or scale bar.

The **count shown in the title** updates in real time.

## The Highlight Dropdown

Use the dropdown to select a specific numbered region. It turns
**red** on the image so you can verify what that object is.
This helps you decide if your size filters are set correctly.

> 💡 **Start with the defaults** and then adjust Min Size upward
> if you see tiny noise spots being counted.

In [ ]:
# Step 5: Inspect detected aggregates with interactive size and shape filters
# Adjust the sliders until the cyan outlines match the bright aggregate spots

# Pre-calculate contours for all regions (done once for speed)
all_contours_precomputed = {}
for p in props:
    region_mask = (watershed_result == p.label)
    all_contours_precomputed[p.label] = measure.find_contours(region_mask, 0.5)

all_labels     = ['None'] + [str(p.label) for p in props]
label_dropdown = Dropdown(options=all_labels, value='None',
                           description='Highlight:',
                           layout=Layout(width='200px'))

outline_plot_output = Output()

def update_outline_plot(min_size, max_size, min_circ, max_circ, highlight_label=None):
    with outline_plot_output:
        clear_output(wait=True)

        props_filtered  = []
        filtered_labels = ['None']
        for p in props:
            circ = (4 * np.pi * p.area) / (p.perimeter ** 2) if p.perimeter > 0 else 0
            if min_size <= p.area <= max_size and min_circ <= circ <= max_circ:
                props_filtered.append(p)
                filtered_labels.append(str(p.label))

        # Update dropdown options without retriggering the callback
        label_dropdown.unobserve(on_label_change, names='value')
        cur = label_dropdown.value
        label_dropdown.options = filtered_labels
        label_dropdown.value   = cur if cur in filtered_labels else 'None'
        label_dropdown.observe(on_label_change, names='value')

        fig, ax = plt.subplots(1, 1, figsize=(9, 9))
        ax.imshow(original_rgb_img)

        for prop in props_filtered:
            contours = all_contours_precomputed.get(prop.label, [])
            is_highlighted = (highlight_label is not None
                              and str(prop.label) == highlight_label)
            col = 'red' if is_highlighted else 'cyan'
            lw  = 2.5   if is_highlighted else 1.0
            for contour in contours:
                ax.plot(contour[:, 1], contour[:, 0], linewidth=lw, color=col)

        ax.set_title(f'Detected Aggregates: {len(props_filtered)}\n'
                     f'(cyan outlines = passing filters)', fontsize=12)
        ax.axis('off')
        plt.tight_layout()
        plt.show()

min_size_slider = IntSlider(value=5,    min=1,   max=500,  step=1,
                             description='Min Size:', continuous_update=True,
                             layout=Layout(width='400px'))
max_size_slider = IntSlider(value=2000, min=10,  max=5000, step=1,
                             description='Max Size:', continuous_update=True,
                             layout=Layout(width='400px'))
min_circ_slider = FloatSlider(value=0.0, min=0.0, max=1.5,  step=0.01,
                               description='Min Circularity:', continuous_update=True,
                               layout=Layout(width='400px'))
max_circ_slider = FloatSlider(value=5.0, min=0.1, max=5.0,  step=0.01,
                               description='Max Circularity:', continuous_update=True,
                               layout=Layout(width='400px'))

def on_slider_change(change):
    update_outline_plot(min_size_slider.value, max_size_slider.value,
                        min_circ_slider.value, max_circ_slider.value,
                        label_dropdown.value)

def on_label_change(change):
    update_outline_plot(min_size_slider.value, max_size_slider.value,
                        min_circ_slider.value, max_circ_slider.value, change.new)

for s in [min_size_slider, max_size_slider, min_circ_slider, max_circ_slider]:
    s.observe(on_slider_change, names='value')
label_dropdown.observe(on_label_change, names='value')

display(VBox([min_size_slider, max_size_slider,
              min_circ_slider, max_circ_slider,
              label_dropdown, outline_plot_output]))
update_outline_plot(min_size_slider.value, max_size_slider.value,
                    min_circ_slider.value, max_circ_slider.value,
                    label_dropdown.value)

# Step 6 — Review and Classify Results

## The Classification Dashboard

This interactive dashboard shows your image alongside three lanes
where every detected region has been automatically sorted:

| Lane | Color | Meaning |
|---|---|---|
| **Accepted** | 🟢 Green | Passed all automatic quality checks — likely a real aggregate |
| **Manual Check** | 🟡 Yellow | Borderline — check these yourself |
| **Rejected** | 🔴 Red | Failed quality checks — likely noise or an artifact |

---

## How to Use It

**To reclassify a card:**
Each card has three buttons — **✅ Accept**, **⚠️ Manual**, and **❌ Reject**.
Click the appropriate button to move that object to a different lane.
The canvas updates immediately to show the new color.

**To inspect an object on the image:**
Click the label and statistics text at the top of any card.
The crosshairs will jump to that object's location on the image
and a white highlight ring will appear around it.

**To zoom in:**
Drag the zoom viewport box to any part of the image for a
magnified view of that region. Drag the crosshairs to move
the zoom target.

---

## When You Are Done

Click the **📋 Lock in Results** button.
This sends your final classification to Python and displays
a table of all accepted aggregates below the dashboard.

**The number of items in the Accepted lane is your aggregate count.**

> 💡 **Suggested workflow:**
> 1. Start with the **Manual Check** lane — these are the borderline
>    cases that need your judgment
> 2. Click each Manual Check card to see it on the image
>    and decide whether it looks like a real aggregate
> 3. Move it to Accepted or Rejected using the buttons
> 4. Briefly scan the Accepted lane for any obvious errors
> 5. Click **📋 Lock in Results** when satisfied
"""

In [ ]:
# Step 6: Classification dashboard
# Click the buttons on each card to move it between lanes
# Click "Lock in Results" when satisfied

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# ── Build data from watershed regions ─────────────────────
raw_data = []
precomputed_contours_json = {}

for p in props:
    region_mask = (watershed_result == p.label)
    contours    = find_contours(region_mask, 0.5)

    if len(contours) == 0:
        vector_perimeter   = 0.0
        raw_contour_points = []
    else:
        contour = max(contours, key=len)
        dx = np.diff(contour[:, 1])
        dy = np.diff(contour[:, 0])
        vector_perimeter   = np.sum(np.sqrt(dx**2 + dy**2))
        raw_contour_points = [[round(float(pt[1]),1), round(float(pt[0]),1)]
                               for pt in contour]

    circularity = ((4 * np.pi * p.area) / (vector_perimeter**2)
                   if vector_perimeter > 0 else 0.0)
    precomputed_contours_json[int(p.label)] = raw_contour_points

    raw_data.append({
        'label':          int(p.label),
        'area_px':        p.area,
        'mean_intensity': round(p.mean_intensity, 1),
        'centroid_y':     round(p.centroid[0], 1),
        'centroid_x':     round(p.centroid[1], 1),
        'circularity':    round(circularity, 3)
    })

df = pd.DataFrame(raw_data)

# ── Calibration ────────────────────────────────────────────
def calculate_otsu_cutoff(values, use_log=False):
    if len(values) < 2:
        return np.mean(values) if len(values) == 1 else 0.0
    wv = np.log10(values + 1e-5) if use_log else np.array(values, dtype=np.float64)
    counts, bins = np.histogram(wv, bins='auto')
    bin_centers  = (bins[:-1] + bins[1:]) / 2
    w1 = np.cumsum(counts); w2 = np.cumsum(counts[::-1])[::-1]
    with np.errstate(divide='ignore', invalid='ignore'):
        m1 = np.cumsum(counts * bin_centers) / w1
        m2 = (np.cumsum((counts * bin_centers)[::-1]) / w2[::-1])[::-1]
        vb = w1[:-1] * w2[1:] * (m1[:-1] - m2[1:])**2
    valid = np.isfinite(vb)
    if not np.any(valid): return np.mean(values)
    cutoff = bin_centers[np.argmax(np.where(valid, vb, -np.inf))]
    return 10**cutoff if use_log else cutoff

OTSU_MIN_AREA = calculate_otsu_cutoff(df['area_px'].values, use_log=True)

if not df.empty:
    si  = np.sort(df['mean_intensity'].values)[::-1]
    tot = np.sum(si)
    if tot > 0:
        cs  = np.cumsum(si) / tot
        idx = min(np.searchsorted(cs, 0.99), len(si)-1)
        CUMULATIVE_MIN_INTENSITY = si[idx]
    else:
        CUMULATIVE_MIN_INTENSITY = 0.0
else:
    CUMULATIVE_MIN_INTENSITY = 0.0

MIN_CIRCULARITY = 0.15;  MAX_CIRCULARITY = 5.00
MAX_AREA        = df['area_px'].max()        if not df.empty else 2000
MAX_INTENSITY   = df['mean_intensity'].max() if not df.empty else 255

# ── Score and classify ─────────────────────────────────────
sync_scores = []; sync_status = []
max_obs_area = df['area_px'].max() if not df.empty else 1

for _, row in df.iterrows():
    area = row['area_px']; mi = row['mean_intensity']; cv = row['circularity']
    if area < OTSU_MIN_AREA:
        sync_scores.append(0.0)
        sync_status.append(f"Rejected (Area {area:.0f}px < Otsu)")
        continue
    elif mi < CUMULATIVE_MIN_INTENSITY:
        sync_scores.append(0.0)
        sync_status.append(f"Rejected (Intensity {mi:.1f} < cutoff)")
        continue
    elif cv < MIN_CIRCULARITY:
        sync_scores.append(0.0); sync_status.append("Rejected (Circularity Too Low)"); continue
    elif cv > MAX_CIRCULARITY:
        sync_scores.append(0.0); sync_status.append("Rejected (Circularity Too High)"); continue
    reasons = []
    if area < OTSU_MIN_AREA * 1.15:
        reasons.append(f"Borderline Area ({area:.0f}px)")
    if mi < CUMULATIVE_MIN_INTENSITY * 1.15:
        reasons.append(f"Borderline Intensity ({mi:.1f})")
    if reasons:
        sync_scores.append(0.0)
        sync_status.append("Manual Check (" + "; ".join(reasons) + ")")
        continue
    a_score = area / max_obs_area
    ic      = (CUMULATIVE_MIN_INTENSITY + MAX_INTENSITY) / 2
    i_score = 1.0 - abs(mi - ic) / max(ic - CUMULATIVE_MIN_INTENSITY, 1)
    cc      = (MIN_CIRCULARITY + MAX_CIRCULARITY) / 2
    c_score = 1.0 - abs(cv - cc) / max(cc - MIN_CIRCULARITY, 1)
    sync_scores.append(round(0.4*a_score + 0.35*i_score + 0.25*c_score, 3))
    sync_status.append("Accepted Aggregate")

df['Aggregate_Score'] = sync_scores
df['Classification']  = sync_status

# ── Convert image for canvas ───────────────────────────────
display_img_array = (original_rgb_img
                     if 'original_rgb_img' in globals()
                     else np.zeros((600,800,3), dtype=np.uint8))
if display_img_array.dtype != np.uint8:
    mn, mx   = display_img_array.min(), display_img_array.max()
    norm_img = (((display_img_array-mn) / max(mx-mn,1)) * 255).astype(np.uint8)
else:
    norm_img = display_img_array

pil_img = Image.fromarray(norm_img)
buff    = io.BytesIO()
pil_img.save(buff, format='JPEG')
image_data_uri = ('data:image/jpeg;base64,'
                  + base64.b64encode(buff.getvalue()).decode('utf-8'))

accepted_df = df[df['Classification'] == 'Accepted Aggregate'].sort_values(
    'Aggregate_Score', ascending=False)
manual_df   = df[df['Classification'].str.contains('Manual Check', na=False)]
rejected_df = df[df['Classification'].str.contains('Rejected',     na=False)]

n_accepted = len(accepted_df)
n_manual   = len(manual_df)
n_rejected = len(rejected_df)

def create_item_list(dataframe, bin_id):
    items = []
    for _, row in dataframe.iterrows():
        lbl = int(row['label'])
        cls = str(row['Classification'])
        reason = (cls.replace('Manual Check (','').replace('Rejected (','').replace(')','')
                  if '(' in cls else '')
        items.append({
            'roi':          str(lbl),
            'area':         int(row['area_px']),
            'mean':         float(row['mean_intensity']),
            'circ':         float(row['circularity']),
            'x':            float(row['centroid_x']),
            'y':            float(row['centroid_y']),
            'reason':       reason,
            'starting_bin': bin_id,
            'contour':      precomputed_contours_json.get(lbl, [])
        })
    return items

all_items_json = json.dumps({
    'accepted': create_item_list(accepted_df, 'bin-accepted'),
    'manual':   create_item_list(manual_df,   'bin-manual'),
    'rejected': create_item_list(rejected_df, 'bin-rejected')
})

# Initial live assignments
live_lane_assignments = {}
for it in create_item_list(accepted_df, 'bin-accepted'):
    live_lane_assignments[int(it['roi'])] = 'bin-accepted'
for it in create_item_list(manual_df,   'bin-manual'):
    live_lane_assignments[int(it['roi'])] = 'bin-manual'
for it in create_item_list(rejected_df, 'bin-rejected'):
    live_lane_assignments[int(it['roi'])] = 'bin-rejected'

live_accepted_df  = accepted_df.copy().reset_index(drop=True)
df_display_handle = IPython.display.DisplayHandle()

# ── Python callback: called once by Lock-in button ────────
def sync_all_assignments(assignments_json):
    global live_accepted_df, live_lane_assignments
    try:
        assignments = json.loads(assignments_json)
        for label_str, lane in assignments.items():
            live_lane_assignments[int(label_str)] = lane
        accepted_ids     = [lbl for lbl, ln in live_lane_assignments.items()
                            if ln == 'bin-accepted']
        live_accepted_df = df[df['label'].isin(accepted_ids)].copy().reset_index(drop=True)
        table_part = (live_accepted_df.to_html(classes='table', index=False)
                      if not live_accepted_df.empty
                      else '<br><i>(No aggregates in Accepted lane)</i>')
        df_display_handle.update(HTML(
            '<div style="margin-top:10px; font-family:Arial; font-size:13px; '
            'padding:12px; background:#e8f5e9; border-radius:6px; '
            'border:2px solid #2ecc71;">'
            '<b style="color:#27ae60;">✅ Results locked in: '
            + str(len(live_accepted_df))
            + ' accepted aggregates</b><br><br>'
            + table_part + '</div>'
        ))
    except Exception as e:
        print(f"Sync error: {e}")

output.register_callback('notebook.SyncAllAssignments', sync_all_assignments)

# ── Scale HUD parameters to image dimensions ──────────────
hud_h, hud_w = original_rgb_img.shape[:2]
shorter_dim  = min(hud_w, hud_h)
zoom_box_size    = int(min(450, max(120, shorter_dim * 0.22)))
zoom_factor      = 4 if shorter_dim >= 1500 else (3 if shorter_dim >= 900 else 2)
crosshair_radius = int(min(30, max(8, shorter_dim * 0.015)))
zb_default_x     = hud_w - zoom_box_size - 20
zb_default_y     = hud_h - zoom_box_size - 20
ch_default_x     = hud_w // 2
ch_default_y     = hud_h // 2

# ── Render HUD ─────────────────────────────────────────────
html_spatial_dashboard = f'''
<style>
  .lane-group {{
    flex: 1;
    display: flex;
    flex-direction: column;
    min-width: 220px;
  }}
  .bin-lane {{
    flex-grow: 1;
    min-height: 200px;
    max-height: 380px;
    overflow-y: auto;
    border-radius: 6px;
    padding: 8px;
    display: flex;
    flex-direction: column;
    gap: 6px;
  }}
  .roi-item-card {{
    background: #fff;
    padding: 6px 10px;
    border-radius: 6px;
    font-size: 11px;
    box-shadow: 0 1px 3px rgba(0,0,0,.08);
    border: 1px solid #ddd;
  }}
  .roi-item-card.highlighted {{
    background: #e3f2fd !important;
    border: 2px solid #2196f3 !important;
  }}
  .card-btn {{
    border: none;
    border-radius: 4px;
    padding: 2px 7px;
    font-size: 10px;
    cursor: pointer;
    font-weight: bold;
    margin-right: 3px;
    margin-top: 4px;
  }}
  .btn-accept  {{ background:#2ecc71; color:#fff; }}
  .btn-manual  {{ background:#f39c12; color:#fff; }}
  .btn-reject  {{ background:#e74c3c; color:#fff; }}
  .btn-accept:hover {{ background:#27ae60; }}
  .btn-manual:hover {{ background:#d68910; }}
  .btn-reject:hover {{ background:#c0392b; }}
</style>

<div style="font-family:Arial; background:#f1f2f6; padding:20px; border-radius:8px;
            max-width:1500px; margin:auto; display:flex; flex-direction:column; gap:16px;">

  <div style="text-align:center;">
    <h3 style="color:#2c3e50; margin:0 0 4px 0;">🎯 Aggregate Classification Dashboard</h3>
    <p style="color:#7f8c8d; margin:0 0 10px 0; font-size:12px;">
      <b>Click ✅ / ⚠️ / ❌ buttons on each card</b> to move it between lanes
      &nbsp;|&nbsp; <b>Click a card</b> to highlight it on the image
    </p>
    <button onclick="syncWithPython()"
            style="background:#27ae60; color:#fff; border:none; border-radius:6px;
                   padding:10px 28px; font-size:14px; font-weight:bold;
                   cursor:pointer; box-shadow:0 2px 6px rgba(0,0,0,.2);">
      📋 Lock in Results
    </button>
    <span id="sync-status"
          style="margin-left:14px; font-size:12px; color:#7f8c8d; font-style:italic;">
      (not yet synced to Python)
    </span>
  </div>

  <div style="display:flex; flex-direction:row; gap:16px;
              align-items:flex-start; flex-wrap:wrap;">

    <!-- Canvas panel -->
    <div style="flex:3; min-width:420px; background:#111; padding:12px;
                border-radius:8px; box-shadow:inset 0 0 10px #000;
                display:flex; justify-content:center; align-items:center;">
      <canvas id="wormCanvas"
              style="display:block; max-width:80%; height:auto;
                     border:1px solid #333; cursor:crosshair;"></canvas>
    </div>

    <!-- Lane panel -->
    <div style="flex:2; min-width:480px; display:flex; gap:10px;
                background:#fff; padding:12px; border-radius:8px;
                border:1px solid #dcdde1; box-shadow:0 3px 6px rgba(0,0,0,.05);">

      <div class="lane-group">
        <h4 style="color:#27ae60; margin:0 0 6px 0; text-align:center; font-size:13px;">
          ✅ Accepted</h4>
        <div id="bin-accepted" class="bin-lane"
             style="background:#e8f5e9; border:2px solid #2ecc71;"></div>
        <div id="accepted-count"
             style="text-align:center; font-size:11px; color:#27ae60;
                    font-weight:bold; margin-top:4px;">Items: 0</div>
      </div>

      <div class="lane-group">
        <h4 style="color:#f39c12; margin:0 0 6px 0; text-align:center; font-size:13px;">
          ⚠️ Manual Check</h4>
        <div id="bin-manual" class="bin-lane"
             style="background:#fff8e1; border:2px solid #f39c12;"></div>
        <div id="manual-count"
             style="text-align:center; font-size:11px; color:#f39c12;
                    font-weight:bold; margin-top:4px;">Items: 0</div>
      </div>

      <div class="lane-group">
        <h4 style="color:#c0392b; margin:0 0 6px 0; text-align:center; font-size:13px;">
          ❌ Rejected</h4>
        <div id="bin-rejected" class="bin-lane"
             style="background:#ffebee; border:2px solid #e74c3c;"></div>
        <div id="rejected-count"
             style="text-align:center; font-size:11px; color:#c0392b;
                    font-weight:bold; margin-top:4px;">Items: 0</div>
      </div>

    </div>
  </div>
</div>

<script>
// ── Core state ─────────────────────────────────────────────
const masterDataMap = new Map();
const baseImage     = new Image();
const canvas        = document.getElementById('wormCanvas');
const ctx           = canvas.getContext('2d');

const ZOOM_FACTOR          = {zoom_factor};
const ZOOM_BOX_SIZE        = {zoom_box_size};
const CROSSHAIR_HIT_RADIUS = {crosshair_radius};

let targetPoint         = {{ x: {ch_default_x}, y: {ch_default_y} }};
let zoomBoxOffset       = {{ x: {zb_default_x}, y: {zb_default_y} }};
let isDraggingCrosshair = false;
let isDraggingZoomBox   = false;
let zoomDragStartOffset = {{ x: 0, y: 0 }};
let lastHighlightId     = null;

// ── Load image ─────────────────────────────────────────────
baseImage.src = "{image_data_uri}";
baseImage.onload = function() {{
  canvas.width  = baseImage.naturalWidth  || 800;
  canvas.height = baseImage.naturalHeight || 600;
  redrawCanvas();
}};

// ── Canvas utilities ───────────────────────────────────────
function getCanvasPoint(e) {{
  const r = canvas.getBoundingClientRect();
  return {{
    x: (e.clientX - r.left) * (canvas.width  / r.width),
    y: (e.clientY - r.top)  * (canvas.height / r.height)
  }};
}}
function isOverCrosshair(cx,cy) {{
  return Math.sqrt((cx-targetPoint.x)**2+(cy-targetPoint.y)**2) <= CROSSHAIR_HIT_RADIUS;
}}
function isOverZoom(cx,cy) {{
  return cx>=zoomBoxOffset.x && cx<=zoomBoxOffset.x+ZOOM_BOX_SIZE &&
         cy>=zoomBoxOffset.y && cy<=zoomBoxOffset.y+ZOOM_BOX_SIZE;
}}

canvas.addEventListener('mousedown', function(e) {{
  const pt = getCanvasPoint(e);
  if (isOverCrosshair(pt.x,pt.y)) {{
    isDraggingCrosshair=true; canvas.style.cursor='grabbing';
  }} else if (isOverZoom(pt.x,pt.y)) {{
    isDraggingZoomBox=true;
    zoomDragStartOffset.x=pt.x-zoomBoxOffset.x;
    zoomDragStartOffset.y=pt.y-zoomBoxOffset.y;
    canvas.style.cursor='move';
  }}
}});
canvas.addEventListener('mousemove', function(e) {{
  const pt = getCanvasPoint(e);
  if (isDraggingCrosshair) {{
    targetPoint.x=Math.max(0,Math.min(canvas.width, pt.x));
    targetPoint.y=Math.max(0,Math.min(canvas.height,pt.y));
    redrawCanvas();
  }} else if (isDraggingZoomBox) {{
    zoomBoxOffset.x=Math.max(0,Math.min(canvas.width-ZOOM_BOX_SIZE,  pt.x-zoomDragStartOffset.x));
    zoomBoxOffset.y=Math.max(0,Math.min(canvas.height-ZOOM_BOX_SIZE, pt.y-zoomDragStartOffset.y));
    redrawCanvas();
  }} else {{
    if      (isOverCrosshair(pt.x,pt.y)) canvas.style.cursor='move';
    else if (isOverZoom(pt.x,pt.y))      canvas.style.cursor='grab';
    else                                  canvas.style.cursor='crosshair';
  }}
}});
canvas.addEventListener('mouseup', function(e) {{
  if (isDraggingCrosshair) {{ isDraggingCrosshair=false; canvas.style.cursor='crosshair'; return; }}
  if (isDraggingZoomBox)   {{ isDraggingZoomBox=false;   canvas.style.cursor='crosshair'; return; }}
  // Click on aggregate to highlight
  const pt=getCanvasPoint(e);
  let matchedId=null, closest=999999;
  masterDataMap.forEach((item,key) => {{
    const d=Math.sqrt((item.x-pt.x)**2+(item.y-pt.y)**2);
    const r=Math.max(5,Math.sqrt(item.area/Math.PI));
    if (d<=r+20 && d<closest) {{ closest=d; matchedId=key; }}
  }});
  if (matchedId) highlightCard(matchedId);
}});
canvas.addEventListener('mouseleave', function() {{
  isDraggingCrosshair=false; isDraggingZoomBox=false;
}});

// ── Drawing ────────────────────────────────────────────────
function drawContour(pts,stroke,lw) {{
  if (!pts||!pts.length) return;
  ctx.beginPath(); ctx.moveTo(pts[0][0],pts[0][1]);
  for (let i=1;i<pts.length;i++) ctx.lineTo(pts[i][0],pts[i][1]);
  ctx.closePath(); ctx.strokeStyle=stroke; ctx.lineWidth=lw; ctx.stroke();
}}
function redrawCanvas() {{
  ctx.clearRect(0,0,canvas.width,canvas.height);
  ctx.drawImage(baseImage,0,0);
  masterDataMap.forEach(item => {{
    let col='rgba(231,76,60,0.5)';
    if      (item.lane==='bin-accepted') col='rgba(46,204,113,0.8)';
    else if (item.lane==='bin-manual')   col='rgba(243,156,18,0.8)';
    // Highlighted item gets a thicker white outline then colored
    if (item.uid===lastHighlightId) {{
      drawContour(item.contour,'white',3.5);
    }}
    drawContour(item.contour,col,1.5);
  }});
  drawCrosshair(); drawZoomViewport();
}}
function drawCrosshair() {{
  ctx.save(); ctx.strokeStyle='#00d2ff'; ctx.lineWidth=1;
  ctx.beginPath(); ctx.moveTo(0,targetPoint.y); ctx.lineTo(canvas.width,targetPoint.y); ctx.stroke();
  ctx.beginPath(); ctx.moveTo(targetPoint.x,0); ctx.lineTo(targetPoint.x,canvas.height); ctx.stroke();
  ctx.restore();
}}
function drawZoomViewport() {{
  ctx.save();
  const bx=zoomBoxOffset.x, by=zoomBoxOffset.y;
  ctx.strokeStyle='#00d2ff'; ctx.lineWidth=1.5; ctx.setLineDash([4,4]);
  ctx.beginPath(); ctx.moveTo(targetPoint.x,targetPoint.y);
  ctx.lineTo(bx+ZOOM_BOX_SIZE/2,by+ZOOM_BOX_SIZE/2); ctx.stroke();
  ctx.setLineDash([]);
  const srcSz=ZOOM_BOX_SIZE/ZOOM_FACTOR;
  ctx.fillStyle='#111'; ctx.fillRect(bx,by,ZOOM_BOX_SIZE,ZOOM_BOX_SIZE);
  ctx.imageSmoothingEnabled=false;
  ctx.drawImage(canvas,targetPoint.x-srcSz/2,targetPoint.y-srcSz/2,
                srcSz,srcSz,bx,by,ZOOM_BOX_SIZE,ZOOM_BOX_SIZE);
  ctx.strokeStyle='rgba(0,210,255,0.85)'; ctx.lineWidth=1;
  ctx.beginPath(); ctx.moveTo(bx,by+ZOOM_BOX_SIZE/2);
  ctx.lineTo(bx+ZOOM_BOX_SIZE,by+ZOOM_BOX_SIZE/2); ctx.stroke();
  ctx.beginPath(); ctx.moveTo(bx+ZOOM_BOX_SIZE/2,by);
  ctx.lineTo(bx+ZOOM_BOX_SIZE/2,by+ZOOM_BOX_SIZE); ctx.stroke();
  ctx.strokeStyle='#00d2ff'; ctx.lineWidth=3;
  ctx.strokeRect(bx,by,ZOOM_BOX_SIZE,ZOOM_BOX_SIZE);
  ctx.fillStyle='#00d2ff'; ctx.font='bold 11px Arial';
  ctx.fillText('ZOOM '+ZOOM_FACTOR+'×  (drag to move)',bx+8,by+18);
  ctx.restore();
}}

// ── Highlight ──────────────────────────────────────────────
function highlightCard(uid) {{
  // Remove previous highlight
  document.querySelectorAll('.roi-item-card').forEach(c=>c.classList.remove('highlighted'));
  const card=document.getElementById(uid);
  if (card) {{
    card.classList.add('highlighted');
    card.scrollIntoView({{behavior:'smooth',block:'nearest'}});
  }}
  const meta=masterDataMap.get(uid);
  if (meta) {{
    lastHighlightId=uid;
    targetPoint.x=meta.x; targetPoint.y=meta.y;
    redrawCanvas();
  }}
}}

// ── Counter update ─────────────────────────────────────────
function updateCounters() {{
  document.getElementById('accepted-count').innerText =
    'Items: '+document.getElementById('bin-accepted').querySelectorAll('.roi-item-card').length;
  document.getElementById('manual-count').innerText =
    'Items: '+document.getElementById('bin-manual').querySelectorAll('.roi-item-card').length;
  document.getElementById('rejected-count').innerText =
    'Items: '+document.getElementById('bin-rejected').querySelectorAll('.roi-item-card').length;
}}

// ── Move card function (replaces drag-and-drop) ───────────
// HTML5 drag-and-drop is unreliable in Colab's sandboxed iframe.
// Instead, each card has small buttons that call this function directly.
// Click events work reliably in all iframe contexts.
function moveCard(uid, targetLaneId) {{
  const card = document.getElementById(uid);
  const lane = document.getElementById(targetLaneId);
  if (!card || !lane) return;

  // Move the card DOM element into the target lane
  lane.appendChild(card);

  // Update the lane border color on the card to match its new home
  const colors = {{
    'bin-accepted': '#2ecc71',
    'bin-manual':   '#f39c12',
    'bin-rejected': '#e74c3c'
  }};
  card.style.borderLeft = '5px solid ' + (colors[targetLaneId] || '#ccc');

  // Update masterDataMap so canvas contours refresh correctly
  const meta = masterDataMap.get(uid);
  if (meta) {{
    meta.lane = targetLaneId;
    redrawCanvas();
  }}

  updateCounters();
}}

// ── Lock-in: bulk sync to Python ──────────────────────────
function syncWithPython() {{
  const assignments = {{}};
  masterDataMap.forEach((item, uid) => {{
    const rawLabel = uid.replace(/^[^-]+-/, '');
    assignments[rawLabel] = item.lane;
  }});
  document.getElementById('sync-status').innerText = 'Syncing...';
  document.getElementById('sync-status').style.color = '#e67e22';
  google.colab.kernel.invokeFunction(
    'notebook.SyncAllAssignments', [JSON.stringify(assignments)], {{}});
  setTimeout(() => {{
    const n = document.getElementById('bin-accepted')
                      .querySelectorAll('.roi-item-card').length;
    document.getElementById('sync-status').innerText = '✅ Locked in: '+n+' accepted';
    document.getElementById('sync-status').style.color = '#27ae60';
  }}, 1200);
}}

// ── Populate lanes ────────────────────────────────────────
(function() {{
  const payload = {all_items_json};

  function populate(items, laneId, prefix, borderColor) {{
    const lane = document.getElementById(laneId);
    items.forEach(item => {{
      const uid = prefix + '-' + item.roi;
      masterDataMap.set(uid, {{
        uid:     uid,
        roi:     item.roi,
        x:       item.x,
        y:       item.y,
        area:    item.area,
        lane:    laneId,
        contour: item.contour
      }});

      const card     = document.createElement('div');
      card.id        = uid;
      card.className = 'roi-item-card';
      card.style.borderLeft = '5px solid ' + borderColor;

      // Card body: label and stats
      const info = document.createElement('div');
      info.style.marginBottom = '3px';
      info.innerHTML =
        '<strong style="color:#000; font-size:12px;">Label ' + item.roi + '</strong>' +
        (item.reason
          ? ' <span style="font-size:10px;color:#999">(' + item.reason + ')</span>'
          : '') +
        '<br><span style="font-size:10px;color:#666;">' +
        '📐 ' + item.area + ' px &nbsp; ' +
        '💡 ' + item.mean.toFixed(1) + ' &nbsp; ' +
        '⭕ ' + item.circ.toFixed(2) + '</span>';

      // Three classification buttons
      const btnRow = document.createElement('div');

      const btnA = document.createElement('button');
      btnA.className   = 'card-btn btn-accept';
      btnA.textContent = '✅ Accept';
      btnA.onclick = function(e) {{
        e.stopPropagation();
        moveCard(uid, 'bin-accepted');
      }};

      const btnM = document.createElement('button');
      btnM.className   = 'card-btn btn-manual';
      btnM.textContent = '⚠️ Manual';
      btnM.onclick = function(e) {{
        e.stopPropagation();
        moveCard(uid, 'bin-manual');
      }};

      const btnR = document.createElement('button');
      btnR.className   = 'card-btn btn-reject';
      btnR.textContent = '❌ Reject';
      btnR.onclick = function(e) {{
        e.stopPropagation();
        moveCard(uid, 'bin-rejected');
      }};

      btnRow.appendChild(btnA);
      btnRow.appendChild(btnM);
      btnRow.appendChild(btnR);

      card.appendChild(info);
      card.appendChild(btnRow);

      // Click the card body (not buttons) to highlight on canvas
      info.style.cursor = 'pointer';
      info.addEventListener('click', () => highlightCard(uid));

      lane.appendChild(card);
    }});
  }}

  populate(payload.accepted, 'bin-accepted', 'acc', '#2ecc71');
  populate(payload.manual,   'bin-manual',   'man', '#f39c12');
  populate(payload.rejected, 'bin-rejected', 'rej', '#e74c3c');
  updateCounters();
}})();
</script>
'''

display(HTML(html_spatial_dashboard))

# Single clean initialisation
df_display_handle.display(HTML(
    '<div style="margin-top:10px; font-family:Arial; font-size:13px; '
    'padding:12px; background:#f8f9fa; border-radius:6px; '
    'border:1px solid #dee2e6; color:#6c757d;">'
    '<i>Use the <b>✅ / ⚠️ / ❌</b> buttons on each card to reclassify, '
    'then click <b>📋 Lock in Results</b> to save your final count.</i></div>'
))

# print(f"Dashboard: {n_accepted} accepted | {n_manual} manual | {n_rejected} rejected")

# Step 7 — Export Your Results

## What Gets Exported

Running the cell below creates a file called `aggregate_data.csv`
containing a row for each **Accepted** aggregate with:

| Column | Meaning |
|---|---|
| `label` | The number assigned to this aggregate |
| `area_px` | Size of the aggregate in pixels |
| `mean_intensity` | Average brightness of the aggregate |
| `circularity` | How round the aggregate is (1.0 = perfect circle) |
| `centroid_x` | Horizontal position in the image (pixels from left) |
| `centroid_y` | Vertical position in the image (pixels from top) |

## Downloading the File

After running the cell:
1. Click the **folder icon (📁)** in the left panel
2. Find `aggregate_data.csv`
3. Click the **three dots (⋮)** and select **Download**

## Recording Your Results

Write down your final aggregate count (the number of items in the
Accepted lane) in your lab notebook. You will compare this count
across different worm strains, ages, or treatments in your analysis.

> 💡 The total count is also printed when you run the export cell.

In [ ]:
# Step 7: Export your accepted aggregate data to a CSV file

if live_accepted_df.empty:
    print("⚠️  No accepted aggregates to export.")
    print("   Make sure you have run Step 6 and that items are in the Accepted lane.")
else:
    # Select and rename columns for a clean export
    export_df = live_accepted_df[[
        'label', 'area_px', 'mean_intensity', 'circularity',
        'centroid_x', 'centroid_y'
    ]].copy()

    output_filename = 'aggregate_data.csv'
    export_df.to_csv(output_filename, index=False)

    print(f"✅ Exported {len(export_df)} accepted aggregates to '{output_filename}'")
    print()
    print(f"  Total aggregate count: {len(export_df)}")
    print(f"  Mean area:             {export_df['area_px'].mean():.1f} px")
    print(f"  Mean intensity:        {export_df['mean_intensity'].mean():.1f}")
    print()
    print("Download the file from the Files panel (📁) on the left.")
    print()
    print("Summary table:")
    display(export_df.describe().round(2))